In [ ]:
import numpy as np
import pyvista as pv
import finitewave as fw


nsub = 7
radius = 15

mesh = pv.Icosphere(nsub=nsub, radius=radius)

coords = mesh.points
elems = mesh.faces.reshape((-1, 4))[:, 1:4]

# find coordinates in spherecial coordinates
theta = np.arctan2(coords[:, 1], coords[:, 0])
phi = np.arctan2(coords[:, 2], np.sqrt(coords[:, 0]**2 + coords[:, 1]**2))

# make holes at (1) z = -radius, (2) theta = 0, phi = pi/4 (3) theta = pi/3, phi = pi/4
center1 = np.array([0, 0, -radius])
center2 = coords[(theta < 0.1) & (theta > -0.1) & (phi < np.pi/4 + 0.1) & (phi > np.pi/4 - 0.1)][0]
center3 = coords[(theta < np.pi + 0.1) & (theta > np.pi - 0.1) & (phi < np.pi/4 + 0.1) & (phi > np.pi/4 - 0.1)][0]

holes_center = [center1, center2, center3]
holes_radius = [radius / 1.5, radius / 2, radius / 2]

mask = np.ones(coords.shape[0], dtype=bool)
for center, r in zip(holes_center, holes_radius):
    dist = np.linalg.norm(coords - center, axis=1)
    mask &= dist > r
    
old_inds = - np.ones(coords.shape[0], dtype=int)
coords = coords[mask, :]
elems = elems[np.all(mask[elems], axis=1), :]

old_inds[mask] = np.arange(coords.shape[0])
elems = old_inds[elems]

grid = fw.PyVistaSurfaceGrid(coords, elems)

pl = pv.Plotter()
pl.add_mesh(grid)
pl.add_points(np.array(holes_center), color='red', point_size=10)
pl.show()

Widget(value='<iframe src="http://localhost:51872/index.html?ui=P_0x36d5a0910_15&reconnect=auto" class="pyvist…